# 02 - Fine-tune Qwen (QLoRA) and evaluate
Runtime -> **T4 GPU**. Same Drive folder as notebook 01.

**W&B key:** Colab left sidebar -> key icon (Secrets) -> add `WANDB_API_KEY`, enable notebook access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/finetuned_llm
!nvidia-smi | head -15

In [ ]:
%%capture
!pip install unsloth
!pip install -q scikit-learn wandb

In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['WANDB_PROJECT'] = 'triage-finetune'
import wandb; wandb.login()

## 1. Train (QLoRA, 2 epochs, ~30-60 min). Loss curves go to W&B.

In [ ]:
!python scripts/train.py --wandb

## 2. Evaluate the fine-tuned model

In [ ]:
!python scripts/evaluate.py --backend hf --adapter outputs/qwen3-4b-triage-lora --name qwen-ft --splits test real_test

## 3. Report + log the comparison table to W&B
This scores every `outputs/preds_*.jsonl` that is present (copy the Claude baseline files into `outputs/` first if you want them in the table).

In [ ]:
!python scripts/report.py --errors

import json, wandb
rows = json.load(open('outputs/report.json'))
run = wandb.init(project='triage-finetune', name='final-comparison', job_type='eval')
cols = ['model','split','n','core_accuracy','severity_macro_f1','high_sev_recall','intent_macro_f1','manufacturer_macro_f1','error_code_acc','json_valid','p50_latency_s','p95_latency_s','cost_per_1k']
run.log({'comparison': wandb.Table(columns=cols, data=[[r[c] for c in cols] for r in rows])})
run.finish()